In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

stations_df = pd.read_csv(r"C:\Users\LIKITH REDDY\Applied data science\Project\stations_data.csv")

In [ ]:
def generate_optimized_training_data(stations_df, days=30, interval_mins=45):
    all_logs = []
    start_date = datetime(2026, 3, 12, 0, 0)
    
    # Calculate intervals per day (e.g., 1440 mins / 45 mins = 32 intervals)
    intervals_per_day = (24 * 60) // interval_mins 
    
    for day in range(days):
        current_day = start_date + timedelta(days=day)
        day_of_week = current_day.weekday()
        is_weekend = 1 if day_of_week >= 5 else 0
        
        for i in range(intervals_per_day):
            current_time = current_day + timedelta(minutes=interval_mins * i)
            hour = current_time.hour
            
            # Logic for Delhi traffic patterns
            if is_weekend:
                intensity = 0.4 + 0.3 * np.sin(np.pi * (hour - 12) / 12)**2
            else:
                if (8 <= hour <= 11) or (17 <= hour <= 21):
                    intensity = 0.82  # Morning/Evening Rush
                elif (0 <= hour <= 5):
                    intensity = 0.12  # Night
                else:
                    intensity = 0.45  # Afternoon
            
            # Adding noise for generalisation
            intensity = np.clip(intensity + np.random.normal(0, 0.07), 0.05, 0.95)

            for _, row in stations_df.iterrows():
                total = int(row['charging_points'])
                occupied = np.random.binomial(total, intensity)
                
                all_logs.append({
                    "timestamp": current_time,
                    "station_id": row['station_id'],
                    "hour": hour,
                    "day_of_week": day_of_week,
                    "is_weekend": is_weekend,
                    "total_slots": total,
                    "occupied_slots": occupied,
                    "free_slots": total - occupied
                })
                
    return pd.DataFrame(all_logs)

# Generate the cleaner, smaller dataset
training_df_45 = generate_optimized_training_data(stations_df, days=30, interval_mins=45)
training_df_45.to_csv("rsu_training_45min.csv", index=False)